# Calculating the Allan Variance for our dataset.

In [1]:
import os
import re
import numpy as np
import pandas as pd
from tqdm.notebook import tqdm
from datetime import datetime as dt

import warnings; warnings.filterwarnings('ignore')

In [2]:
DATA_LOC = '../../data'

# path to processed and ready data
DATA_PATH = os.path.join(DATA_LOC, 'lme_ready')

# path to save data to on completion
VIS_PATH = os.path.join(DATA_LOC, 'web_vis')

# post processing & ready for data visualization steps
FINAL_DOC = os.path.join(DATA_LOC, 'resids.parquet')
print(os.path.exists(FINAL_DOC))

True


In [ ]:
# allan variance outputs
allan_var = 'allan_var-difs.parquet'

#### Load data

In [ ]:
# df = pd.read_table(FINAL_DOC, sep='\t')
df = pd.read_parquet(FINAL_DOC)
df.shape

In [ ]:
df['conversation_id'].nunique()

In [ ]:
df.head()

In [ ]:
df.shape

#### Merge on Self vs. Other and create $\Delta(o-s)$ column

In [ ]:
merge_columns = ['groups', 'sample_delta']
value_columns = ['Hxy']

In [ ]:
df = pd.merge(
    left=df.loc[df['self']], left_on=merge_columns,
    right=df[merge_columns+value_columns].loc[~df['self']], right_on=merge_columns,
    how='left'
)
df.shape

In [ ]:
df.isna().sum()

In [ ]:
df['Hxy'] = df['Hxy_y'] - df['Hxy_x']

In [ ]:
del df['Hxy_x']
del df['Hxy_y']

In [ ]:
df.head()

In [ ]:
print(df.shape)
df = df.loc[~df['Hxy'].isna()]
df.shape

### Merge `copied doc` Method.

In [ ]:
merge_columns = ['groups', 'sample_delta']

In [ ]:
value_columns = ['Hxy']

#### First turn after the current turn

In [ ]:
# make a copy of our df
df_ = df.copy()

# for every turn_delta, subtract it by one (so 2 => 1, et al.)
df_['sample_delta'] -= 1

In [ ]:
# merge the copied doc with the actual doc using dyad ID, self, 
#   and turn_delta as indexes
df = pd.merge(
    left=df, left_on=merge_columns,
    right=df_[merge_columns+value_columns], right_on=merge_columns,
    how='left'
)

# check the shape
print(df.shape)
df.isna().sum()

#### Second turn after the current turn

In [ ]:
# for every turn_delta, subtract it by one (so 2 => 1, et al.)
df_['sample_delta'] -= 1

In [ ]:
# merge the copied doc with the actual doc using dyad ID, self, 
#   and turn_delta as indexes
df = pd.merge(
    left=df, left_on=merge_columns,
    right=df_[merge_columns+value_columns], right_on=merge_columns,
    how='left'
)

# check the shape
print(df.shape)
df.isna().sum()

#### Final checks

In [ ]:
# check what row x column combos have NaNs
df.isna().sum()

In [ ]:
# get a sample of the NaN containing rows
df[['groups', 'sample_delta', 'Hxy_x', 'Hxy_y', 'Hxy']].loc[df['Hxy'].isna()].sample(n=10).head(n=10)

In [ ]:
df = df.loc[~df['Hxy'].isna()]
df.shape

In [ ]:
df = df.rename(columns={'Hxy_x': 'resid', 'Hxy_y': 'resid_1', 'Hxy': 'resid_2', 'resid': 'residual'})

In [ ]:
df[['groups', 'resid', 'resid_1', 'resid_2']].head()

### Create Allan Variance Scale Parameter

In [ ]:
group_cols = ['groups', 'self']
number_cols = ['sample_delta']

In [ ]:
N = df[group_cols+number_cols].groupby(by=group_cols).agg('count').reset_index()
N = N.rename(columns={'sample_delta': 'N'})

In [ ]:
df = pd.merge(
    left=df, left_on=group_cols,
    right=N, right_on=group_cols,
    how='left'
)
df.shape

In [ ]:
tau = df[group_cols+number_cols].groupby(by=group_cols).agg('max').reset_index()
tau = tau.rename(columns={'sample_delta': 'tau'})

In [ ]:
df = pd.merge(
    left=df, left_on=group_cols,
    right=tau, right_on=group_cols,
    how='left'
)
df.shape

In [ ]:
df.isna().sum()

In [ ]:
df.head()

## Calculate Allan $\sigma^2$

In [ ]:
df = df.loc[~df['resid_2'].isna()]

In [ ]:
df['allan_var'] = (1/(2*(df['tau']**2))) * ((df['resid_2'] - (2*df['resid_1']) + df['resid'])**2)

In [ ]:
df.head()

### Save outputs

In [ ]:
# df.to_csv(
#     allan_var,
#     index=False,
#     encoding='utf-8',
#     sep='\t'
# )

In [ ]:
df.to_parquet(allan_var)